# Oscilloscope connection pipeline

This notebook walks through the normal RIGOL DHO workflow: choose a VISA resource, connect, verify `*IDN?`, configure waveform readout, capture one or more channels, save CSV data, plot the result, and close the connection cleanly.

Run it from inside the `hardware-capture` repository. Keep hardware-specific SCPI settings close to the setup cell so the capture cells stay simple.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "hardware_capture" / "oscilloscope").exists():
            return path
    raise RuntimeError("Run this notebook from inside the hardware-capture repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

## Import the oscilloscope utilities

The public API keeps connection handling in `RigolDHO`, capture parameters in `WaveformCaptureConfig`, and CSV output in `save_waveform_csv`.

In [ ]:
from hardware_capture.oscilloscope import (
    DEFAULT_IP,
    RigolDHO,
    WaveformCaptureConfig,
    default_visa_resource,
    parse_time_seconds,
    save_waveform_csv,
)

DEFAULT_IP

## Define the VISA resource

Use `TCPIP::<ip>::INSTR` for the normal VXI-11/LXI path. If your lab uses a raw socket endpoint, build it with `default_visa_resource(ip, socket=True)` instead.

In [ ]:
SCOPE_IP = DEFAULT_IP
VISA_RESOURCE = default_visa_resource(SCOPE_IP)
TIMEOUT_MS = 30_000

CHANNELS = ["CHAN1"]
WINDOW_START = "-10us"
WINDOW_END = "30us"

VISA_RESOURCE

## Connect and verify `*IDN?`

Keep the `scope` object open while you run the capture cells. The final cell closes it explicitly.

In [ ]:
scope = RigolDHO(resource=VISA_RESOURCE, timeout_ms=TIMEOUT_MS)
scope.open()

identity = scope.query("*IDN?")
print(identity)

## Configure waveform readout

This selects the channel source and ASCII RAW waveform transfer. Add lab-specific SCPI commands here if the timebase, trigger source, coupling, or vertical scale should be controlled from the notebook.

In [ ]:
for channel in CHANNELS:
    scope.write(f":WAV:SOUR {channel}")
scope.write(":WAV:MODE RAW")
scope.write(":WAV:FORM ASC")

print("Trigger status:", scope.trigger_status())

## Build capture settings

The time window is converted to scope point indices after the preamble is read. `allow_clamp=True` keeps a slightly oversized window from failing when the acquired record is shorter than expected.

In [ ]:
def make_config(channel: str) -> WaveformCaptureConfig:
    return WaveformCaptureConfig(
        channel=channel,
        window_start_s=parse_time_seconds(WINDOW_START),
        window_end_s=parse_time_seconds(WINDOW_END),
        chunk_points=100_000,
        allow_clamp=True,
    )


configs = {channel: make_config(channel) for channel in CHANNELS}
configs

## Capture waveform data

`capture_waveform` stops the scope before reading by default, then reads the requested point window in chunks.

In [ ]:
waveforms = {}
for channel, config in configs.items():
    waveform = scope.capture_waveform(config)
    waveforms[channel] = waveform
    print(
        f"{channel}: {len(waveform.voltage)} samples, "
        f"{waveform.sample_rate:.6g} Sa/s, "
        f"points {waveform.start_point} to {waveform.stop_point}"
    )

## Save captured waveforms

`save_waveform_csv` writes local files and automatically avoids overwriting an existing path by adding a numbered suffix.

In [ ]:
output_dir = REPO_ROOT / "captures"
saved_paths = {}

for channel, waveform in waveforms.items():
    path = output_dir / f"scope_{channel.lower()}_window.csv"
    saved_paths[channel] = save_waveform_csv(
        path,
        waveform,
        metadata={
            "scope": waveform.source,
            "channel": waveform.channel,
            "start_point": waveform.start_point,
            "stop_point": waveform.stop_point,
        },
    )
    print(saved_paths[channel])

## Plot the waveform

This quick plot is usually enough to confirm the trigger timing, selected channel, and voltage scale before moving to analysis.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for channel, waveform in waveforms.items():
    ax.plot(waveform.time * 1e6, waveform.voltage, label=channel, linewidth=1.0)

ax.set_xlabel("Time (us)")
ax.set_ylabel("Voltage (V)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Optional repeated SINGLE capture

Set `REPEAT_SHOTS` above zero when you want the scope to arm in SINGLE mode for repeated externally triggered captures. Leave it at zero for normal notebook bring-up.

In [ ]:
REPEAT_SHOTS = 0
repeated_paths = []

for shot_index in range(REPEAT_SHOTS):
    waveform = scope.capture_single_waveform(
        make_config(CHANNELS[0]),
        start_shot=lambda: print("Scope armed; start the external trigger source now."),
        trigger_timeout_s=30.0,
        rearm_after_read=False,
    )
    path = output_dir / f"scope_{CHANNELS[0].lower()}_shot{shot_index + 1:04d}.csv"
    repeated_paths.append(save_waveform_csv(path, waveform))

repeated_paths

## Close the connection

Always close the session when the notebook is done so the VISA resource is released for the next tool.

In [ ]:
scope.close()